<a href="https://colab.research.google.com/github/EnricoKiin/ids-project-d1-nypd/blob/main/Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup

In [1]:
#Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotnine as p9

In [2]:
import os
from google.colab import drive
drive.mount('/content/drive')
os.chdir("/content/drive/My Drive/Colab Notebooks/IDS/")

Mounted at /content/drive


In [16]:
nypdArrestDF = pd.read_parquet("arrests_correct_types.parquet")
nypdComplaintDF = pd.read_parquet("complaint_correct_types.parquet")

#Cleaning

## Arrest data

In [17]:
nypdArrestDF.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6264978 entries, 0 to 6264977
Data columns (total 18 columns):
 #   Column             Dtype         
---  ------             -----         
 0   ARREST_KEY         int64         
 1   ARREST_DATE        datetime64[ns]
 2   PD_CD              category      
 3   PD_DESC            string        
 4   KY_CD              category      
 5   OFNS_DESC          category      
 6   LAW_CODE           category      
 7   LAW_CAT_CD         category      
 8   ARREST_BORO        category      
 9   ARREST_PRECINCT    category      
 10  JURISDICTION_CODE  category      
 11  AGE_GROUP          category      
 12  PERP_SEX           category      
 13  PERP_RACE          category      
 14  X_COORD_CD         float64       
 15  Y_COORD_CD         float64       
 16  Latitude           float64       
 17  Longitude          float64       
dtypes: category(11), datetime64[ns](1), float64(4), int64(1), string(1)
memory usage: 412.4 MB


# NYPD Arrest Data: Cleaning Checklist

| Done | #  | Column            | Dtype          | Notes |
|:----:|---:|-------------------|----------------|-------|
| [ ]  | 0  | ARREST_KEY        | int64          |       |
| [ ]  | 1  | ARREST_DATE       | datetime64[ns] |       |
| [ ]  | 2  | PD_CD             | category       |       |
| [ ]  | 3  | PD_DESC           | string         |       |
| [ ]  | 4  | KY_CD             | category       |       |
| [ ]  | 5  | OFNS_DESC         | category       |       |
| [ ]  | 6  | LAW_CODE          | category       |       |
| [X]  | 7  | LAW_CAT_CD        | category       |       |
| [X]  | 8  | ARREST_BORO       | category       |       |
| [X]  | 9  | ARREST_PRECINCT   | category       |       |
| [X]  | 10 | JURISDICTION_CODE | category       |       |
| [X]  | 11 | AGE_GROUP         | category       |       |
| [X]  | 12 | PERP_SEX          | category       |       |
| [X]  | 13 | PERP_RACE         | category       |       |
| [ ]  | 14 | X_COORD_CD        | float64        |       |
| [ ]  | 15 | Y_COORD_CD        | float64        |       |
| [ ]  | 16 | Latitude          | float64        |       |
| [ ]  | 17 | Longitude         | float64        |       |

In [21]:
"""
print(nypdArrestDF["AGE_GROUP"].unique().tolist()) #Real are 45-64, 25-44, 18-24, <18 and 65+
print(nypdArrestDF["PERP_SEX"].unique())
print(nypdArrestDF["PERP_RACE"].unique())
"""



nypdArrestDF["AGE_GROUP"] = nypdArrestDF["AGE_GROUP"].str.strip().astype("category")
nypdArrestDF["AGE_GROUP"] = nypdArrestDF["AGE_GROUP"].where(nypdArrestDF["AGE_GROUP"].isin(["45-64", "25-44", "<18", "18-24", "65+"]))

nypdArrestDF["PERP_SEX"] = nypdArrestDF["PERP_SEX"].replace("U", np.nan)
nypdArrestDF["PERP_RACE"] = nypdArrestDF["PERP_RACE"].replace(["UNKNOWN", "OTHER"], np.nan)

/tmp/ipykernel_593/445271127.py:12: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
/tmp/ipykernel_593/445271127.py:13: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.


In [ ]:
#Got this list from https://www.nyc.gov/site/nypd/bureaus/patrol/precincts-landing.page
valid_precincts = [
    # Manhattan
    "1", "5", "6", "7", "9", "10", "13", "14", "17", "18", "19", "20",
    "22", "23", "24", "25", "26", "28", "30", "32", "33", "34",
    # Bronx
    "40", "41", "42", "43", "44", "45", "46", "47", "48", "49", "50", "52",
    # Brooklyn
    "60", "61", "62", "63", "66", "67", "68", "69", "70", "71", "72", "73",
    "75", "76", "77", "78", "79", "81", "83", "84", "88", "90", "94",
    # Queens
    "100", "101", "102", "103", "104", "105", "106", "107", "108", "109",
    "110", "111", "112", "113", "114", "115", "116",
    # Staten Island
    "120", "121", "122", "123",
]
#27th precinct does not exist it is fictual and so is 483
#18th is actualy Midtown North now
#14th is Midtown South Precinct
#22nd is Central Park


"""
with pd.option_context('display.max_rows', None):
  print(nypdArrestDF["ARREST_BORO"].value_counts()) #All fine
  print(nypdArrestDF["ARREST_PRECINCT"].value_counts())
  print(nypdArrestDF["JURISDICTION_CODE"].value_counts()) #Did not find any info that any of the values are invalid
"""
nypdArrestDF["ARREST_PRECINCT"] = nypdArrestDF["ARREST_PRECINCT"].where(nypdArrestDF["ARREST_PRECINCT"].isin(valid_precincts))
nypdArrestDF["ARREST_PRECINCT"] = nypdArrestDF["ARREST_PRECINCT"].cat.remove_unused_categories() #Removing 483 and 27 from the categories since they are now NaN

In [25]:
print(nypdArrestDF["LAW_CAT_CD"].value_counts()) #M-misdemeanor, F-Felony, I-Infraction, V-violation
#9 is probably meant to be 'I' but we don't know

nypdArrestDF["LAW_CAT_CD"] = nypdArrestDF["LAW_CAT_CD"].replace("9", np.nan)

LAW_CAT_CD
M    4027384
F    1879253
V     301786
I      27406
Name: count, dtype: int64


In [26]:
with pd.option_context('display.max_rows', None):
  print(nypdArrestDF["OFNS_DESC"].value_counts())

OFNS_DESC
DANGEROUS DRUGS                                1166438
ASSAULT 3 & RELATED OFFENSES                    684955
PETIT LARCENY                                   333901
OTHER OFFENSES RELATED TO THEFT                 332592
FELONY ASSAULT                                  311398
VEHICLE AND TRAFFIC LAWS                        258876
MISCELLANEOUS PENAL LAW                         258611
OTHER STATE LAWS                                255158
DANGEROUS WEAPONS                               243758
ROBBERY                                         213353
CRIMINAL TRESPASS                               212048
OTHER TRAFFIC INFRACTION                        185841
GRAND LARCENY                                   174656
OFFENSES AGAINST PUBLIC ADMINISTRATION          156544
POSSESSION OF STOLEN PROPERTY 5                 149877
CRIMINAL MISCHIEF & RELATED OFFENSES            146039
INTOXICATED & IMPAIRED DRIVING                  114597
FORGERY                                         104351


## Complaint data